**Manual Annotation**

This notebook contains the interface for manual annotation of the dataset, to compare the results of the model with the human annotations.

In [ ]:
TO_ANNOTATE = "topics"  # Change this to 'moral_foundations' or 'stance' as needed
TO_SAMPLE = 200  # Number of ads to annotate

## Initialization

Run these cells to initialize the environment and load the dataset.

In [ ]:
from hydra import compose, initialize
from omegaconf import DictConfig

with initialize(config_path="configs", job_name="dem_analysis", version_base="1.3"):
    cfg: DictConfig = compose(config_name="config")

In [ ]:
import time
from dataclasses import dataclass

from IPython.display import display
from ipywidgets import HTML, Button, HBox, Output, VBox

with open(cfg.topics_path) as f:
    topics = [line.strip() for line in f.readlines()]

moral_foundations = [
    "care/harm",
    "fairness/cheating",
    "loyalty/betrayal",
    "authority/subversion",
    "sanctity/degradation",
    "liberty/oppression",
]

stance = ["for", "against"]

results = []
global_config = {"current_index": 0}


@dataclass
class Annotation:
    id: int
    topic: str
    body: str


ad_text = HTML()
button_out = Output()


def annotate_next_ad(df):
    if global_config["current_index"] >= len(df):
        button_out.clear_output()
        ad_text.value = "All ads have been annotated. Run the next cell to save the results."
        return

    ad = df.iloc[global_config["current_index"]]["ad_creative_bodies"]

    ad_text.value = (
        f"<b>Ad {global_config['current_index'] + 1} of {len(df)}</b><p style='white-space: pre-wrap'>{ad}</p>"
    )


def on_topic_selected(topic, df):
    results.append(
        Annotation(
            id=df.index[global_config["current_index"]],
            topic=topic,
            body=df.iloc[global_config["current_index"]]["ad_creative_bodies"],
        )
    )
    global_config["current_index"] += 1
    annotate_next_ad(df)


def create_buttons(items, df):
    buttons = []

    def make_click_handler(topic):
        def on_click(b):
            on_topic_selected(topic, df)
            b.button_style = "success"
            time.sleep(0.5)
            b.button_style = "info"

        return on_click

    for topic in items:
        btn = Button(description=topic, button_style="info", layout={"width": "200px"})
        btn._click_handlers.callbacks.clear()

        btn.on_click(make_click_handler(topic))
        buttons.append(btn)

    button_rows = [HBox(buttons[i : i + 4]) for i in range(0, len(buttons), 4)]

    with button_out:
        display(VBox(button_rows))

    annotate_next_ad(df)


annotation_targets = {
    "topics": topics,
    "moral_foundations": moral_foundations,
    "stance": stance,
}


def reset_state():
    ad_text.value = ""
    global_config["current_index"] = 0
    results.clear()


def annotate(df):
    button_out.clear_output()
    create_buttons(annotation_targets[TO_ANNOTATE], df)
    print("Welcome to the annotation tool! Please select the most relevant topic for each ad.")
    display(VBox([button_out, ad_text]))

Load data...

In [ ]:
import random

import pandas as pd

random.seed(42)
df = pd.read_csv(cfg.data.path)

df_clustered = df[df["cluster"] > -1].copy()
df_unclustered = df[df["cluster"] == -1].copy()

annotated_sample = df_clustered.sample(n=TO_SAMPLE, random_state=42).reset_index(drop=True)
unannotated_sample = df_unclustered.sample(n=TO_SAMPLE, random_state=42).reset_index(drop=True)

## Annotation

Run the cells in this section to start the annotation process. The interface will display a random sample from the dataset, and you can provide your annotations for each sample. The annotations will be saved to a CSV file for later analysis.

In [ ]:
ANNOTATE_CLUSTERED = True  # Set to False to annotate unclustered ads

if ANNOTATE_CLUSTERED:
    to_annotate_df = annotated_sample
else:
    to_annotate_df = unannotated_sample

Restart the annotation process by rerunning the following cell.

In [ ]:
assert TO_ANNOTATE in annotation_targets, f"Invalid target: {TO_ANNOTATE}, must be one of {annotation_targets.keys()}"

reset_state()
print(f"Reset progress. Run the next cell to start annotating ads on {TO_ANNOTATE}.")

Run the following cell to start annotating the dataset:

In [ ]:
annotate(to_annotate_df)

Run the following cell to save your annotations:

In [ ]:
from pathlib import Path

assert len(results) > 0, "No annotations were made. Please annotate at least one ad."

indices = [result.id for result in results]
topic = [result.topic for result in results]
bodies = [to_annotate_df["ad_creative_bodies"][i] for i in indices]

timestamp = time.strftime("%Y-%m-%d_%H-%M-%S", time.localtime())
result_df = pd.DataFrame(
    {
        "body": bodies,
        "topic": topic,
    },
    index=indices,
)

annotations_dir = Path(cfg.annotation_path)
save_file = annotations_dir / f"manual_{TO_ANNOTATE}_{timestamp}.csv"
annotations_dir.mkdir(parents=True, exist_ok=True)

result_df.to_csv(save_file)
print(f"Annotations saved to {save_file}")